In [29]:
import onnxruntime as ort
import numpy as np 
import json 
import os

In [30]:
# Start the onnx runtime session: 
session = ort.InferenceSession(
    "../artifacts/latest/models/model.onnx",
    providers=["CPUExecutionProvider"]
)

In [ ]:
# Show this entrie process with a sample example using the min,max formula itself (not the scaler saved.)

# also discuss the timestamps sequence concern. 

# 6 features, for 10 timestamps 
# Based min,max per feature --> normalize these
# create the final input for inference (1, 10, 6) 

In [31]:
# Get inputs ready to pass to onnx session: 
X_inputs = np.load("../artifacts/latest/training_set.npy") 
X_inputs.shape

(5369, 10, 5)

In [32]:
X_inputs[:1]

array([[[0.01960784, 0.95133642, 0.72562175, 0.05263158, 0.99281472],
        [0.        , 0.95133642, 0.72562175, 0.05263158, 0.99281472],
        [0.        , 0.81917363, 0.44198849, 0.05263158, 0.99273165],
        [0.01960784, 0.81917363, 0.30049354, 0.05263158, 0.04381775],
        [0.        , 0.81917363, 0.30049354, 0.05263158, 0.04381775],
        [0.        , 0.81917363, 0.2763037 , 0.05263158, 0.06425219],
        [0.        , 0.81917363, 0.27461721, 0.05263158, 0.00490094],
        [0.        , 0.81917363, 0.27293532, 0.05263158, 0.00490094],
        [0.        , 0.81917363, 0.27293532, 0.05263158, 0.00490094],
        [0.        , 0.81917363, 0.27124423, 0.05023923, 0.00490094]]])

In [33]:
# Pick first sample to pass 
onnx_input = X_inputs[:1].astype(np.float32)
print(onnx_input.shape)

(1, 10, 5)


In [34]:
# get the input_name expected by onnx for input: 
input_name = session.get_inputs()[0].name  # this is "input_sequence" 

In [35]:
# Pass input to onnx session and run the prediction
pred_onnx = session.run(None,{input_name: onnx_input})[0]

In [36]:
pred_onnx.shape

(1, 10, 5)

In [37]:
pred_onnx  # Show the 5th timestamp's load prediction

array([[[-0.01330389,  0.81237864,  0.2805318 ,  0.05534223,
          0.10340524],
        [-0.02879003,  0.80411625,  0.31897026,  0.05437123,
          0.17510845],
        [-0.03267706,  0.8114045 ,  0.34899935,  0.05394526,
          0.22717346],
        [-0.02910696,  0.82264614,  0.36064795,  0.05299177,
          0.24806023],
        [-0.02099523,  0.827205  ,  0.34724775,  0.05163037,
          0.2275393 ],
        [-0.01480778,  0.82432014,  0.31509525,  0.05069766,
          0.17390124],
        [-0.01083814,  0.81611115,  0.27771038,  0.05076435,
          0.10746132],
        [-0.00553325,  0.8066541 ,  0.24749726,  0.05190111,
          0.04789379],
        [ 0.0042861 ,  0.7991224 ,  0.2301327 ,  0.05375348,
          0.00604953],
        [ 0.0187384 ,  0.7946972 ,  0.22471422,  0.05575543,
         -0.01637761]]], dtype=float32)

In [39]:
# Compare based on the upper and lower limit saved in meatadat file: 

# Load metadata file: 
with open('../artifacts/latest/metadata.json') as f:
    metadata = json.load(f)

# Get the limits: 
upper_limit = metadata["upper_limit"]
lower_limit = metadata["lower_limit"]

print(upper_limit)
print(lower_limit)

# Compare against the predicted value: 
pred_value = pred_onnx[:,4,0]



0.13840367093848477
-0.044330755623450645
